# Mini Project: End-to-End Machine Learning Model Development & Deployment

## Project Overview:
This project implements an end-to-end Machine Learning pipeline to predict House Prices based on structural and demographic features:
1. **Data Set Preparation**: Synthetic dataset generation, missing value handling, and feature engineering.
2. **Model Building**: Training Linear Regression, Random Forest, and Gradient Boosting Regressors.
3. **Model Evaluation**: Assessing MAE, RMSE, and R2 scores.
4. **Model Optimization**: Hyperparameter tuning using Grid Search Cross-Validation.
5. **Application Development & Deployment**: Packaging model as a REST API with Flask.
6. **Frontend Integration**: Web-based user interface for real-time interactive valuation.
7. **Dockerization & Testing**: Containerization with Docker and automated REST API verification.

## 1. Data Set Preparation & Exploration

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Generate synthetic realistic housing dataset (1,200 properties)
n_samples = 1200
num_rooms = np.random.choice([1, 2, 3, 4, 5, 6], size=n_samples, p=[0.05, 0.20, 0.40, 0.20, 0.10, 0.05])
num_bathrooms = np.clip(num_rooms - np.random.choice([0, 1, 2], size=n_samples, p=[0.5, 0.4, 0.1]), 1, 5)
area_sqft = np.random.normal(loc=num_rooms * 500 + 400, scale=300, size=n_samples).astype(int)
area_sqft = np.clip(area_sqft, 450, 6000)
location = np.random.choice(['Downtown', 'Suburbs', 'Urban', 'Rural'], size=n_samples, p=[0.25, 0.35, 0.25, 0.15])
age_years = np.random.randint(0, 50, size=n_samples)
has_garage = np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7])

# Location pricing multipliers
loc_multiplier = {'Downtown': 1.6, 'Urban': 1.25, 'Suburbs': 1.1, 'Rural': 0.8}
loc_effect = np.array([loc_multiplier[l] for l in location])

# Ground truth price calculation with realistic non-linear relationships
base_price = 50000
price = (
    base_price +
    (area_sqft * 160 * loc_effect) +
    (num_rooms * 20000) +
    (num_bathrooms * 14000) -
    (age_years * 1200) +
    (has_garage * 22000) +
    np.random.normal(0, 18000, size=n_samples)
)
price = np.round(np.clip(price, 40000, 1500000), 2)

# Create DataFrame
df_raw = pd.DataFrame({
    'num_rooms': num_rooms,
    'num_bathrooms': num_bathrooms,
    'area_sqft': area_sqft.astype(float),
    'location': location,
    'age_years': age_years.astype(float),
    'has_garage': has_garage,
    'price': price
})

# Introduce synthetic missing values to simulate real-world data collection issues
missing_indices_area = np.random.choice(n_samples, size=35, replace=False)
missing_indices_age = np.random.choice(n_samples, size=25, replace=False)
df_raw.loc[missing_indices_area, 'area_sqft'] = np.nan
df_raw.loc[missing_indices_age, 'age_years'] = np.nan

df_raw.to_csv('data/raw_house_data.csv', index=False)
print("Raw housing data generated and saved to data/raw_house_data.csv")
print(f"Dataset shape: {df_raw.shape}")
print("Missing values per column:\n", df_raw.isnull().sum())
df_raw.head()

Raw housing data generated and saved to data/raw_house_data.csv
Dataset shape: (1200, 7)
Missing values per column:
 num_rooms         0
num_bathrooms     0
area_sqft        35
location          0
age_years        25
has_garage        0
price             0
dtype: int64


,num_rooms,num_bathrooms,area_sqft,location,age_years,has_garage,price
0,3,2,1731.0,Urban,6.0,1,479271.18
1,6,5,3337.0,Urban,39.0,0,832006.24
2,4,4,1894.0,Downtown,32.0,1,657007.24
3,3,3,1658.0,Urban,20.0,1,480056.02
4,2,2,1689.0,Suburbs,NaN,0,408244.06


## 2. Data Cleaning & Feature Engineering

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load dataset
df = pd.read_csv('data/raw_house_data.csv')

# Handle missing values
imputer_area = SimpleImputer(strategy='median')
df['area_sqft'] = imputer_area.fit_transform(df[['area_sqft']])

imputer_age = SimpleImputer(strategy='median')
df['age_years'] = imputer_age.fit_transform(df[['age_years']])

# Feature Engineering
df['total_rooms'] = df['num_rooms'] + df['num_bathrooms']
df['room_to_area_ratio'] = df['num_rooms'] / df['area_sqft']

df.to_csv('data/processed_house_data.csv', index=False)
print("Data preprocessed and saved to data/processed_house_data.csv")
print("Remaining null values:", df.isnull().sum().sum())
df.head()

Data preprocessed and saved to data/processed_house_data.csv
Remaining null values: 0


,num_rooms,num_bathrooms,area_sqft,location,age_years,has_garage,price,total_rooms,room_to_area_ratio
0,3,2,1731.0,Urban,6.0,1,479271.18,5,0.001733
1,6,5,3337.0,Urban,39.0,0,832006.24,11,0.001798
2,4,4,1894.0,Downtown,32.0,1,657007.24,8,0.002112
3,3,3,1658.0,Urban,20.0,1,480056.02,6,0.001809
4,2,2,1689.0,Suburbs,25.0,0,408244.06,4,0.001184


## 3. Model Development & Evaluation
We train and compare:
- Linear Regression (Baseline)
- Random Forest Regressor
- Gradient Boosting Regressor

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Define feature sets
feature_cols = ['num_rooms', 'num_bathrooms', 'area_sqft', 'location', 'age_years', 'has_garage', 'total_rooms', 'room_to_area_ratio']
target_col = 'price'

X = df[feature_cols]
y = df[target_col]

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Preprocessing Pipeline
numeric_features = ['num_rooms', 'num_bathrooms', 'area_sqft', 'age_years', 'has_garage', 'total_rooms', 'room_to_area_ratio']
categorical_features = ['location']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"[{name}] MAE: ${mae:,.2f} | RMSE: ${rmse:,.2f} | R2: {r2:.4f}")

df_results = pd.DataFrame(results)

# Plot Model Comparison
fig, ax1 = plt.subplots(figsize=(9, 5))
x_indices = np.arange(len(df_results))
width = 0.35

rects1 = ax1.bar(x_indices - width/2, df_results['MAE'], width, label='MAE ($)', color='#4361ee')
rects2 = ax1.bar(x_indices + width/2, df_results['RMSE'], width, label='RMSE ($)', color='#3a0ca3')

ax1.set_ylabel('Error in USD ($)')
ax1.set_title('Model Performance Comparison (MAE vs RMSE)')
ax1.set_xticks(x_indices)
ax1.set_xticklabels(df_results['Model'])
ax1.legend()
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=300)
plt.close()
print("Model comparison plot saved to outputs/model_comparison.png")

/Users/mukilan/Build-and-Deploy-Machine-Learning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/mukilan/Build-and-Deploy-Machine-Learning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/mukilan/Build-and-Deploy-Machine-Learning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


[Linear Regression] MAE: $28,620.21 | RMSE: $40,934.21 | R2: 0.9591
[Random Forest] MAE: $24,171.62 | RMSE: $36,584.86 | R2: 0.9673


[Gradient Boosting] MAE: $24,167.00 | RMSE: $34,537.14 | R2: 0.9709


Model comparison plot saved to outputs/model_comparison.png


## 4. Hyperparameter Optimization via Grid Search
We optimize the top-performing model (Gradient Boosting Regressor) using 5-Fold Cross-Validation.

In [4]:
from sklearn.model_selection import GridSearchCV
import joblib

gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__max_depth': [3, 4],
    'regressor__subsample': [0.8, 1.0]
}

print("Running Grid Search CV...")
grid_search = GridSearchCV(gb_pipe, param_grid=param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Optimal Parameters: {grid_search.best_params_}")

# Evaluate best model
y_pred_opt = best_model.predict(X_test)
opt_mae = mean_absolute_error(y_test, y_pred_opt)
opt_rmse = root_mean_squared_error(y_test, y_pred_opt)
opt_r2 = r2_score(y_test, y_pred_opt)

print(f"Tuned Model Performance:")
print(f"MAE: ${opt_mae:,.2f}")
print(f"RMSE: ${opt_rmse:,.2f}")
print(f"R2 Score: {opt_r2:.4f}")

# Save the finalized model pipeline for deployment
joblib.dump(best_model, 'models/best_house_price_model.pkl')
joblib.dump(best_model, 'app/model.pkl')
print("Optimized pipeline saved to models/best_house_price_model.pkl and app/model.pkl")

Running Grid Search CV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits


Optimal Parameters: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 4, 'regressor__n_estimators': 200, 'regressor__subsample': 0.8}
Tuned Model Performance:
MAE: $21,834.29
RMSE: $31,733.54
R2 Score: 0.9754
Optimized pipeline saved to models/best_house_price_model.pkl and app/model.pkl


## 5. Model Diagnostics & Visualizations

In [5]:
# 1. Actual vs Predicted Scatter Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_opt, alpha=0.6, color='#4361ee', edgecolors='k', s=45)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.title(f'Actual vs. Predicted House Prices (R2 = {opt_r2:.4f})')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/actual_vs_predicted.png', dpi=300)
plt.close()

# 2. Residual Distribution Plot
residuals = y_test - y_pred_opt
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='#7209b7', bins=30)
plt.axvline(x=0, color='r', linestyle='--')
plt.title('Prediction Residuals Distribution')
plt.xlabel('Residual ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('outputs/residual_distribution.png', dpi=300)
plt.close()

# 3. Feature Importance Plot
trained_regressor = best_model.named_steps['regressor']
cat_encoder = best_model.named_steps['preprocessor'].named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + cat_features

importances = trained_regressor.feature_importances_
feat_df = pd.DataFrame({'Feature': all_features, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='Blues_r')
plt.title('Feature Importances (Gradient Boosting Regressor)')
plt.xlabel('Relative Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=300)
plt.close()
print("All visualization artifacts generated.")

All visualization artifacts generated.


/var/folders/qg/2l8949_j0jdd9374hflg77dr0000gn/T/ipykernel_7317/693714530.py:35: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=feat_df, x='Importance', y='Feature', palette='Blues_r')


## 6. Application Testing (REST API Verification)
We spin up the Flask application locally in the background, send verification requests to both `/health` and `/predict`, and evaluate the responses.

In [6]:
import subprocess
import time
import requests
import sys

# Launch Flask app from app/ directory
flask_server = subprocess.Popen([sys.executable, 'app.py'], cwd='app')
time.sleep(2.5)

try:
    # 1. Health check
    res_health = requests.get('http://127.0.0.1:5000/health')
    print("Health Check Response:", res_health.json())
    assert res_health.status_code == 200

    # 2. Prediction check
    sample_payload = {
        'num_rooms': 4,
        'num_bathrooms': 3,
        'area_sqft': 2500,
        'location': 'Downtown',
        'age_years': 5,
        'has_garage': 1
    }
    res_predict = requests.post('http://127.0.0.1:5000/predict', json=sample_payload)
    print("Prediction API Response:", res_predict.json())
    assert res_predict.status_code == 200
    assert 'predicted_price' in res_predict.json()
    print("Application verification tests passed successfully!")
finally:
    flask_server.terminate()
    flask_server.wait()
    print("Test server shutdown cleanly.")

/Users/mukilan/Build-and-Deploy-Machine-Learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


 * Serving Flask app 'app'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.31.21:5000
Press CTRL+C to quit


Health Check Response: {'service': 'House Price Prediction API', 'status': 'healthy'}
Prediction API Response: {'features': {'age_years': 5.0, 'area_sqft': 2500.0, 'has_garage': True, 'location': 'Downtown', 'num_bathrooms': 3.0, 'num_rooms': 4.0}, 'formatted_price': '$820,470.40', 'predicted_price': 820470.4, 'status': 'success'}
Application verification tests passed successfully!
Test server shutdown cleanly.


127.0.0.1 - - [27/Oct/2026 02:06:13] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2026 02:06:13] "POST /predict HTTP/1.1" 200 -
